---
---
# __Table of Contents__


## __Data Cleaning & Fixing__

### __1. Library Imports__
* __Data Manipulation:__ `pandas` and `numpy` to structure and clean your numbers.
* __Data Encoding (MLOps Standard):__ `OneHotEncoder` from `sklearn.preprocessing` to transform high-cardinality text categories into production-ready numbers while memorizing the data schema.

### __2. Value Standardization & Text Normalization__
1. Working on `train.csv` dataset
2. Working on `test.csv` dataset 
---
---

### __Data Dictionary__

|Variable	     |Definition	     |Key                                             |
|:---------------|:------------------|:-----------------------------------------------|
|passenger_id    |Passenger ID       |
|survival  	     |Survival	         |0 = No, 1 = Yes                                 |
|ticket_class    |Ticket class       |1 = 1st, 2 = 2nd, 3 = 3rd                       |
|name            |Name of person     |
|sex	         |Sex	             |
|age	         |Age in years       |	
|siblings_spouses|# of siblings / spouses aboard the Titanic|
|parents_children|# of parents / children aboard the Titanic|
|ticket	         |Ticket number      |
|fare	         |Passenger fare     |	
|cabin	         |Cabin number       |	
|boarding_port   |Port of Embarkation|	C = Cherbourg, Q = Queenstown, S = Southampton|


### __Variable Notes__

__ticket_class:__ A proxy for socio-economic status (SES)
* 1st = Upper
* 2nd = Middle
* 3rd = Lower

__age:__ Age is fractional if less than 1. If the age is estimated, is it in the form of xx.5

__siblings_spouses:__ The dataset defines family relations in this way...
* Sibling = brother, sister, stepbrother, stepsister
* Spouse = husband, wife (mistresses and fiancés were ignored)

__parents_children:__ The dataset defines family relations in this way...
* Parent = mother, father
* Child = daughter, son, stepdaughter, stepson
* Some children travelled only with a nanny, therefore parch=0 for them.

### __1. Library Imports__
---

In [1]:
# Data manipulation and calculations
import pandas as pd
import numpy as np

# Data encoding
from sklearn.preprocessing import OneHotEncoder

### __2. Structural Gaps (Handling Missing Data)__
---

### __2.1 Working On `train.csv` Dataset__

#### __Stage 1 -->__

#### __Load the Dataset__

In [2]:
# Import the 'train.csv' dataset from 2-under-process folder
train_df = pd.read_csv('../data/2-under-process/train.csv')

In [3]:
# Show the first 5 rows of the dataset
train_df.head()

,passenger_id,survival,ticket_class,name,sex,age,siblings_spouses,parents_children,ticket,fare,cabin,boarding_port
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,Unknown,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,Unknown,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,Unknown,S


In [4]:
# Get the structural breakdown of the dataset
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   passenger_id      891 non-null    int64  
 1   survival          891 non-null    int64  
 2   ticket_class      891 non-null    int64  
 3   name              891 non-null    object 
 4   sex               891 non-null    object 
 5   age               714 non-null    float64
 6   siblings_spouses  891 non-null    int64  
 7   parents_children  891 non-null    int64  
 8   ticket            891 non-null    object 
 9   fare              891 non-null    float64
 10  cabin             891 non-null    object 
 11  boarding_port     891 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


#### 📝 Analysis Note: Categorical Text Conversion Strategy
* __`sex` (Binary Mapping):__ _Convert using explicit mapping._ Since this column contains exactly two categories (`male` and `female`), it will be transformed into a clean binary toggle switch (`0` for male, `1` for female). This approach is fast, avoids creating a fake hierarchy, and scales perfectly without unnecessary column expansion.
  
* __`boarding_port` (One-Hot Encoding):__ _Convert using Scikit-Learn's `OneHotEncoder`._ Because this column contains three distinct geographic options (`S`, `C`, `Q`), sequential numbering would trick the model into assuming an inaccurate numerical ranking. Expanding this column into three separate binary columns guarantees mathematical neutrality and ensures production stability.

In [5]:
# Convet the values of 'sex' into numbers
# 1. Define the universal binary map
gender_encoding = {'male': 1, 'female': 0}

# 2. Safely reassign the column data across both datasets 
train_df['sex'] = train_df['sex'].map(gender_encoding)

# 3. Quick pandas verification check to confirm the data types have locked into numbers
print("--- Post-Encoding Verification ---")
train_df['sex'].dtypes
train_df['sex'].value_counts()

--- Post-Encoding Verification ---


sex
1    577
0    314
Name: count, dtype: int64

In [6]:
# Convet the values of 'boarding_port' into numbers
# 1. Initialize the encoder
encoder = OneHotEncoder(sparse_output=False)

# 2. Extract and transform the 'boarding_port' column
encoded_array = encoder.fit_transform(train_df[['boarding_port']])

# 3. Turn the array into a temporary DataFrame with clean names (boarding_port_C, boarding_port_Q, boarding_port_S)
encoded_cols = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(['boarding_port']))

# 4. Drop the 'boarding_port' column from train_df and join the new numeric columns
train_df = train_df.drop(columns=['boarding_port']).reset_index(drop=True)
train_df = train_df.join(encoded_cols)

# 5. Convert all column headers to lowercase
train_df.columns = train_df.columns.str.lower()

# 6. Check the data
train_df.head()

,passenger_id,survival,ticket_class,name,sex,age,siblings_spouses,parents_children,ticket,fare,cabin,boarding_port_c,boarding_port_q,boarding_port_s
0,1,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,Unknown,0.0,0.0,1.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,C85,1.0,0.0,0.0
2,3,1,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.9250,Unknown,0.0,0.0,1.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1,0,113803,53.1000,C123,0.0,0.0,1.0
4,5,0,3,"Allen, Mr. William Henry",1,35.0,0,0,373450,8.0500,Unknown,0.0,0.0,1.0


#### __Save the Changes__

In [7]:
# Save the data into under-process folder
train_df.to_csv('../data/2-under-process/train.csv', index=False)